# Chapter 5 — Activation, Initialization, Normalization

Runnable companion to `docs/04_activation_initialization_normalization.md`.
Four experiments, each with a "before / after" comparison:

1. Plot five activations and their derivatives on the same figure.
2. Vanishing gradient on a 20-layer sigmoid MLP.
3. Xavier (Glorot) vs. Kaiming (He) initialization on the same ReLU MLP.
4. BatchNorm on/off — convergence speed and final loss.

Generated by `src/build_chapter_04_activation_init_norm.py`. Edit the
builder, not this notebook.

## Setup

In [1]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device("cpu")    # all experiments are CPU-fast
print("torch", torch.__version__, "device", DEVICE)

torch 2.12.0+cu130 device cpu


## 1. Activation curves — value and derivative

Five activations side by side. The *derivative* plot is what matters for
gradient flow: a derivative that lives near `0` for most of its input range
will vanish gradients in deep stacks.

In [2]:
z = torch.linspace(-5, 5, 400)

def deriv(fn, x):
    x = x.detach().clone().requires_grad_(True)
    y = fn(x)
    y.sum().backward()
    return x.grad


acts = {
    "sigmoid":    torch.sigmoid,
    "tanh":       torch.tanh,
    "ReLU":       lambda x: F.relu(x),
    "LeakyReLU":  lambda x: F.leaky_relu(x, negative_slope=0.1),
    "GELU":       lambda x: F.gelu(x),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name, fn in acts.items():
    ax1.plot(z, fn(z), label=name)
    ax2.plot(z, deriv(fn, z), label=name)
ax1.set_title("activation value");  ax1.set_xlabel("z"); ax1.legend()
ax2.set_title("activation derivative"); ax2.set_xlabel("z"); ax2.legend()
ax1.axhline(0, color="gray", lw=0.5); ax2.axhline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/tmp/ipykernel_185636/2501132463.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Reading the plot.**

- *Sigmoid* derivative peaks at `0.25` — every layer multiplies gradient by
  at most `0.25`. Stack 20 layers and the gradient is `<= 0.25^20 ≈ 9e-13`.
- *Tanh* derivative peaks at `1.0` and decays — less vanishing but still
  not great for deep networks.
- *ReLU* derivative is `1` for `z > 0` (clean gradient flow) and `0` for
  `z <= 0` (the "dead ReLU" problem).
- *LeakyReLU* keeps a small slope below zero so no neuron is fully dead.
- *GELU* is smooth — modern Transformers use it for that reason.

## 2. Vanishing gradient demo

Build a 20-layer MLP twice: once with **sigmoid**, once with **ReLU**.
Run a single forward+backward with random inputs and measure the gradient
norm at *each* layer. The sigmoid net's gradients collapse exponentially
toward the input; the ReLU net's gradients stay roughly stable.

In [3]:
class DeepMLP(nn.Module):
    def __init__(self, num_layers=20, width=64, activation=nn.Sigmoid):
        super().__init__()
        layers = [nn.Linear(64, width), activation()]
        for _ in range(num_layers - 2):
            layers += [nn.Linear(width, width), activation()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


def grad_norms_per_layer(model):
    norms = []
    for layer in model.net:
        if isinstance(layer, nn.Linear) and layer.weight.grad is not None:
            norms.append(layer.weight.grad.norm().item())
    return norms


torch.manual_seed(0)
x = torch.randn(32, 64)
y = torch.randn(32, 1)

results = {}
for name, act in [("sigmoid", nn.Sigmoid), ("ReLU", nn.ReLU)]:
    torch.manual_seed(0)
    model = DeepMLP(num_layers=20, activation=act)
    loss = F.mse_loss(model(x), y)
    loss.backward()
    results[name] = grad_norms_per_layer(model)

fig, ax = plt.subplots(figsize=(9, 4))
for name, norms in results.items():
    ax.semilogy(range(len(norms)), norms, marker="o", label=name)
ax.set_xlabel("layer index (input -> output)")
ax.set_ylabel("‖∂L/∂W‖  (log scale)")
ax.set_title("Gradient norm per layer in a 20-layer MLP")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

print("sigmoid gradient ratio (first / last layer):",
      f"{results['sigmoid'][0] / results['sigmoid'][-1]:.2e}")
print("ReLU    gradient ratio (first / last layer):",
      f"{results['ReLU'][0] / results['ReLU'][-1]:.2e}")

sigmoid gradient ratio (first / last layer): 3.54e-17
ReLU    gradient ratio (first / last layer): 3.23e-07


/tmp/ipykernel_185636/1781673488.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The sigmoid network's early layers see gradient many orders of magnitude
smaller than the last layer — they cannot learn anything. The ReLU network
stays within ~one order of magnitude end-to-end.

## 3. Xavier vs. Kaiming initialization

Two ReLU MLPs, one initialized with **Xavier-normal** (designed for tanh),
one with **Kaiming-normal** (designed for ReLU). Train both on a synthetic
regression task. The Kaiming-init network should train noticeably faster.

In [4]:
def init_mlp(model, scheme):
    for layer in model.net:
        if isinstance(layer, nn.Linear):
            if scheme == "xavier":
                nn.init.xavier_normal_(layer.weight)
            elif scheme == "kaiming":
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)


def train_quick(scheme, steps=600, lr=0.01):
    torch.manual_seed(0)
    model = DeepMLP(num_layers=10, width=64, activation=nn.ReLU)
    init_mlp(model, scheme)
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    torch.manual_seed(1)
    X = torch.randn(256, 64)
    y_target = (X.sum(dim=1, keepdim=True) * 0.1).sin()

    history = []
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(model(X), y_target)
        loss.backward()
        opt.step()
        history.append(loss.item())
    return history


hist_xavier  = train_quick("xavier")
hist_kaiming = train_quick("kaiming")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_xavier,  label="Xavier (Glorot) init")
ax.plot(hist_kaiming, label="Kaiming (He) init")
ax.set_xlabel("step"); ax.set_ylabel("MSE loss")
ax.set_title("Same ReLU MLP, two initializations")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"final loss Xavier:  {hist_xavier[-1]:.5f}")
print(f"final loss Kaiming: {hist_kaiming[-1]:.5f}")

final loss Xavier:  0.03601
final loss Kaiming: 0.00448


/tmp/ipykernel_185636/1182896631.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. BatchNorm on / off

Same 10-layer ReLU MLP again, this time with BatchNorm inserted after each
linear (before activation). BatchNorm should converge faster and to a lower
loss, especially at the higher learning rate where the no-BN version may
diverge or plateau.

In [5]:
class DeepMLP_BN(nn.Module):
    def __init__(self, num_layers=10, width=64, use_bn=True):
        super().__init__()
        in_dim = 64
        layers = []
        for i in range(num_layers - 1):
            layers.append(nn.Linear(in_dim if i == 0 else width, width))
            if use_bn:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


def train_bn(use_bn, steps=600, lr=0.05):
    torch.manual_seed(0)
    model = DeepMLP_BN(use_bn=use_bn)
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    torch.manual_seed(1)
    X = torch.randn(256, 64)
    y = (X.sum(dim=1, keepdim=True) * 0.1).sin()

    history = []
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(model(X), y)
        loss.backward()
        opt.step()
        history.append(loss.item())
    return history


hist_no_bn = train_bn(use_bn=False)
hist_bn    = train_bn(use_bn=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_no_bn, label="no BatchNorm")
ax.plot(hist_bn,    label="with BatchNorm")
ax.set_xlabel("step"); ax.set_ylabel("MSE loss")
ax.set_title("10-layer ReLU MLP, SGD lr=0.05")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"final loss no-BN: {hist_no_bn[-1]:.5f}")
print(f"final loss BN:    {hist_bn[-1]:.5f}")

final loss no-BN: 0.35352
final loss BN:    0.00000


/tmp/ipykernel_185636/2023885696.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


At `lr=0.05`, the no-BN network is on the edge of stability — the loss
oscillates and barely descends. The BN network absorbs the larger learning
rate easily and reaches a much lower loss.

## Self-test

<details>
<summary>Q1 — Why is ReLU paired with Kaiming init, not Xavier?</summary>

ReLU zeroes half the activations, halving the variance. Kaiming compensates
with `√(2/n_in)` (vs. Xavier's `√(1/n_in)`). With Xavier init, ReLU
activations collapse layer by layer.
</details>

<details>
<summary>Q2 — During inference, what statistics does BatchNorm use?</summary>

The running mean and variance accumulated during training, not the
current batch's statistics. `model.eval()` flips this switch.
</details>